In [2]:
import requests

url = "https://datos.madrid.es/api/3/action/datastore_search"

params = {
    "resource_id": "202087-1-trafico-intensidad",
    "limit": 5
}

response = requests.get(url, params=params)

print("Status:", response.status_code)
print(response.text[:500])

Status: 404
<!DOCTYPE html>
<!-- saved from url=(0064)https://servpub.madrid.es/mantenimiento/indexServicioMejora.html -->
<html lang="es"><head><meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
  
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Ayuntamiento de Madrid - En mantenimiento</title>
  <meta name="description" content="PÃ¡gina de mantenimiento del Ayuntamiento de Madrid">

  <style>
    :root {
      --madrid-blue: #0038A8;        /* Pantone 286


In [3]:
url = "https://datos.madrid.es/dataset/202087-0-trafico-intensidad/resource/202087-0-trafico-intensidad/download/202087-0-trafico-intensidad.xml"

response = requests.get(url)

print("Status:", response.status_code)
print(response.text[:1000])

Status: 200
<pms>
  <fecha_hora>26/08/2026 19:35:04</fecha_hora>
  <pm>
    <idelem>9841</idelem>
    <descripcion>Valle de Mena S-E - Acc.Ramon Castroviejo-Gta.Isaac Rabín</descripcion>
    <accesoAsociado>0301005</accesoAsociado>
    <intensidad>660</intensidad>
    <ocupacion>7</ocupacion>
    <carga>12</carga>
    <nivelServicio>0</nivelServicio>
    <intensidadSat>3100</intensidadSat>
    <error>N</error>
    <subarea>0328</subarea>
    <st_x>438339,375874991</st_x>
    <st_y>4480454,96970565</st_y>
  </pm>
  <pm>
    <idelem>9843</idelem>
    <descripcion>Dr.Ramon Castroviejo O-E - Av.Miraflores-San Martin de Porres</descripcion>
    <accesoAsociado>0301003</accesoAsociado>
    <intensidad>999</intensidad>
    <ocupacion>27</ocupacion>
    <carga>60</carga>
    <nivelServicio>2</nivelServicio>
    <intensidadSat>2000</intensidadSat>
    <error>N</error>
    <subarea>0328</subarea>
    <st_x>438098,880458143</st_x>
    <st_y>4480455,13738494</st_y>
  </pm>
  <pm>


In [4]:
import xml.etree.ElementTree as ET

root = ET.fromstring(response.content)

registros = []

fecha_hora = root.find("fecha_hora").text

for pm in root.findall("pm"):
    fila = {
        "fecha_hora": fecha_hora
    }

    for campo in pm:
        fila[campo.tag] = campo.text

    registros.append(fila)

df_trafico = pd.DataFrame(registros)

df_trafico.head()

,fecha_hora,idelem,descripcion,accesoAsociado,intensidad,ocupacion,carga,nivelServicio,intensidadSat,error,subarea,st_x,st_y,velocidad
0,26/08/2026 19:35:04,9841,Valle de Mena S-E - Acc.Ramon Castroviejo-Gta....,0301005,660,7,12,0,3100,N,0328,"438339,375874991","4480454,96970565",NaN
1,26/08/2026 19:35:04,9843,Dr.Ramon Castroviejo O-E - Av.Miraflores-San M...,0301003,999,27,60,2,2000,N,0328,"438098,880458143","4480455,13738494",NaN
2,26/08/2026 19:35:04,9842,Nueva Zelanda S-N - Isla Cristina-Gta.Isaac Rabín,0301001,140,2,18,0,700,N,0328,"438305,950926946","4480365,97449961",NaN
3,26/08/2026 19:35:04,11396,Dr. Ramon Castroviejo E-O - Islas Bikini-Gta I...,0301004,680,0,17,0,3150,N,0328,"438345,801923105","4480548,97911408",NaN
4,26/08/2026 19:35:04,11243,Islas Aleutianas N-S - Manuel Garrido -Isaac R...,0301002,240,1,12,0,1350,N,0328,"438226,45123268","4480578,31957825",NaN


In [5]:
df_trafico.shape

(4893, 14)

In [6]:
df_trafico.columns.tolist()

['fecha_hora',
 'idelem',
 'descripcion',
 'accesoAsociado',
 'intensidad',
 'ocupacion',
 'carga',
 'nivelServicio',
 'intensidadSat',
 'error',
 'subarea',
 'st_x',
 'st_y',
 'velocidad']

In [7]:
df_trafico.dtypes

fecha_hora        str
idelem            str
descripcion       str
accesoAsociado    str
intensidad        str
ocupacion         str
carga             str
nivelServicio     str
intensidadSat     str
error             str
subarea           str
st_x              str
st_y              str
velocidad         str
dtype: object

In [8]:
# Convertir fecha
df_trafico["fecha_hora"] = pd.to_datetime(
    df_trafico["fecha_hora"],
    format="%d/%m/%Y %H:%M:%S"
)

# Columnas que deben ser numéricas
columnas_numericas = [
    "intensidad",
    "ocupacion",
    "carga",
    "nivelServicio",
    "intensidadSat",
    "velocidad"
]

# Convertir columnas a números
for columna in columnas_numericas:
    df_trafico[columna] = pd.to_numeric(
        df_trafico[columna],
        errors="coerce"
    )

# Convertir coordenadas
df_trafico["st_x"] = pd.to_numeric(
    df_trafico["st_x"].str.replace(",", ".", regex=False),
    errors="coerce"
)

df_trafico["st_y"] = pd.to_numeric(
    df_trafico["st_y"].str.replace(",", ".", regex=False),
    errors="coerce"
)

# Comprobar resultado
df_trafico.dtypes

fecha_hora        datetime64[us]
idelem                       str
descripcion                  str
accesoAsociado               str
intensidad                 int64
ocupacion                  int64
carga                      int64
nivelServicio              int64
intensidadSat            float64
error                        str
subarea                      str
st_x                     float64
st_y                     float64
velocidad                float64
dtype: object

In [9]:
# Control de calidad de los datos

print("VALORES NULOS:")
print(df_trafico.isnull().sum())

print("\nFILAS DUPLICADAS:")
print(df_trafico.duplicated().sum())

print("\nIDELEM DUPLICADOS:")
print(df_trafico["idelem"].duplicated().sum())

VALORES NULOS:
fecha_hora           0
idelem               0
descripcion         27
accesoAsociado     737
intensidad           0
ocupacion            0
carga                0
nivelServicio        0
intensidadSat      310
error               27
subarea            310
st_x                 0
st_y                 0
velocidad         4583
dtype: int64

FILAS DUPLICADAS:
0

IDELEM DUPLICADOS:
0


In [10]:
# Porcentaje de valores nulos por columna

porcentaje_nulos = (
    df_trafico.isnull().mean() * 100
).round(2)

porcentaje_nulos.sort_values(ascending=False)

velocidad         93.66
accesoAsociado    15.06
intensidadSat      6.34
subarea            6.34
error              0.55
descripcion        0.55
ocupacion          0.00
intensidad         0.00
fecha_hora         0.00
idelem             0.00
carga              0.00
nivelServicio      0.00
st_x               0.00
st_y               0.00
dtype: float64

In [11]:
# Eliminar la columna velocidad por tener un 93,66 % de valores nulos
df_trafico = df_trafico.drop(columns=["velocidad"])

# Comprobar el resultado
print("Dimensiones:", df_trafico.shape)
print("\nNulos restantes:")
print(df_trafico.isnull().sum())

Dimensiones: (4893, 13)

Nulos restantes:
fecha_hora          0
idelem              0
descripcion        27
accesoAsociado    737
intensidad          0
ocupacion           0
carga               0
nivelServicio       0
intensidadSat     310
error              27
subarea           310
st_x                0
st_y                0
dtype: int64


In [12]:
# Revisar los valores de la columna error

print("Valores de error:")
print(df_trafico["error"].value_counts(dropna=False))

print("\nPorcentaje:")
print(
    df_trafico["error"]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

Valores de error:
error
N      4493
S       373
NaN      27
Name: count, dtype: int64

Porcentaje:
error
N      91.83
S       7.62
NaN     0.55
Name: proportion, dtype: float64


In [13]:
# Mantener únicamente mediciones sin error
df_trafico_limpio = df_trafico[
    df_trafico["error"] == "N"
].copy()

# Comprobar resultado
print("Dataset original:", df_trafico.shape)
print("Dataset limpio:", df_trafico_limpio.shape)

print("\nValores de error:")
print(df_trafico_limpio["error"].value_counts())

Dataset original: (4893, 13)
Dataset limpio: (4493, 13)

Valores de error:
error
N    4493
Name: count, dtype: int64


In [14]:
# Estadísticas básicas de las principales variables de tráfico

columnas_analisis = [
    "intensidad",
    "ocupacion",
    "carga",
    "nivelServicio",
    "intensidadSat"
]

df_trafico_limpio[columnas_analisis].describe()

,intensidad,ocupacion,carga,nivelServicio,intensidadSat
count,4493.000000,4493.000000,4493.000000,4493.000000,4210.000000
mean,396.430892,5.907857,21.089695,0.314489,1983.603325
std,519.438977,12.083745,18.072839,0.594691,1143.098918
min,-1.000000,-1.000000,-1.000000,-1.000000,50.000000
25%,100.000000,1.000000,8.000000,0.000000,1200.000000
50%,240.000000,2.000000,17.000000,0.000000,1500.000000
75%,480.000000,5.000000,29.000000,1.000000,3000.000000
max,6420.000000,100.000000,274.000000,3.000000,7200.000000


In [15]:
# Contar valores -1 en las principales variables

columnas_analisis = [
    "intensidad",
    "ocupacion",
    "carga",
    "nivelServicio",
    "intensidadSat"
]

for columna in columnas_analisis:
    cantidad = (df_trafico_limpio[columna] == -1).sum()
    print(f"{columna}: {cantidad}")

intensidad: 11
ocupacion: 11
carga: 11
nivelServicio: 21
intensidadSat: 0


In [16]:
import numpy as np

columnas_con_menos_uno = [
    "intensidad",
    "ocupacion",
    "carga",
    "nivelServicio"
]

df_trafico_limpio[columnas_con_menos_uno] = (
    df_trafico_limpio[columnas_con_menos_uno]
    .replace(-1, np.nan)
)

# Comprobar resultado
print("Valores -1 restantes:")
print((df_trafico_limpio[columnas_con_menos_uno] == -1).sum())

print("\nNulos actuales:")
print(df_trafico_limpio[columnas_con_menos_uno].isnull().sum())

Valores -1 restantes:
intensidad       0
ocupacion        0
carga            0
nivelServicio    0
dtype: int64

Nulos actuales:
intensidad       11
ocupacion        11
carga            11
nivelServicio    21
dtype: int64


In [17]:
# Traducir el código de nivel de servicio a una categoría comprensible

mapa_nivel_servicio = {
    0: "Fluido",
    1: "Lento",
    2: "Retenciones",
    3: "Congestión"
}

df_trafico_limpio["estado_trafico"] = (
    df_trafico_limpio["nivelServicio"]
    .map(mapa_nivel_servicio)
)

In [18]:
df_trafico_limpio[
    ["nivelServicio", "estado_trafico"]
].value_counts(dropna=False)

nivelServicio  estado_trafico
0.0            Fluido            3285
1.0            Lento              982
2.0            Retenciones        163
3.0            Congestión          42
NaN            NaN                 21
Name: count, dtype: int64

In [19]:
# Mostrar los puntos con congestión

congestionados = (
    df_trafico_limpio[
        df_trafico_limpio["estado_trafico"] == "Congestión"
    ]
    .sort_values("carga", ascending=False)
)

congestionados[
    ["idelem", "descripcion", "intensidad", "ocupacion", "carga", "nivelServicio"]
].head(10)

,idelem,descripcion,intensidad,ocupacion,carga,nivelServicio
2481,4465,(MICRO) Pl. San Juan de la Cruz N-S (P.M.3)(De...,972.0,23.0,119.0,3.0
1007,10332,(MICRO) PSO. SATA. MARIA DE LA CABEZA S-N,1500.0,33.0,100.0,3.0
1151,10293,(TACTICO) SALCEDO S-N (SALVATIERRA - LEZAMA),0.0,100.0,100.0,3.0
2320,4339,PRINCESA N-S(GIRO IZQDA. FERNANDO EL CATOLICO),0.0,100.0,100.0,3.0
2954,3678,(TACTICO) Juan Sanchez E-O (Candido Mateos - C...,0.0,100.0,100.0,3.0
1061,10500,VIA LUSITANA E-O Ø154 (PL. RENDICION BREDA-AV....,0.0,100.0,100.0,3.0
1008,10848,(MICRO) VIA LUSITANA O-E (A 4 mts LINEA DETENC...,0.0,100.0,100.0,3.0
1003,10616,FERREIRA N-S (CASTRO ORO - VALLE ORO),0.0,100.0,100.0,3.0
795,4966,(TACTICO) EXPROPIACION S-N (GRAN AVENIDA - AV....,0.0,98.0,98.0,3.0
1749,7030,(MICRO) Pl. Colón (Delante G. 7) - (MICRO) Pl....,684.0,11.0,98.0,3.0


In [20]:
# Distribución porcentual del estado del tráfico

distribucion_estado = (
    df_trafico_limpio["estado_trafico"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

distribucion_estado

estado_trafico
Fluido         73.46
Lento          21.96
Retenciones     3.64
Congestión      0.94
Name: proportion, dtype: float64

In [21]:
import sqlite3

# Crear/conectar con la base de datos
conexion = sqlite3.connect("trafico_madrid.db")

print("Conexión con SQLite creada correctamente")

Conexión con SQLite creada correctamente


In [22]:
# Crear la tabla principal de tráfico

conexion.execute("""
CREATE TABLE IF NOT EXISTS trafico (
    idelem TEXT,
    fecha_hora TEXT,
    descripcion TEXT,
    accesoAsociado TEXT,
    intensidad REAL,
    ocupacion REAL,
    carga REAL,
    nivelServicio REAL,
    intensidadSat REAL,
    error TEXT,
    subarea TEXT,
    st_x REAL,
    st_y REAL,
    estado_trafico TEXT,
    PRIMARY KEY (idelem, fecha_hora)
)
""")

conexion.commit()

print("Tabla 'trafico' creada correctamente")

Tabla 'trafico' creada correctamente


In [23]:
# Guardar la primera captura en SQLite

df_trafico_limpio.to_sql(
    "trafico",
    conexion,
    if_exists="append",
    index=False
)

conexion.commit()

print("Datos guardados correctamente en SQLite")

Datos guardados correctamente en SQLite


In [24]:
# Comprobar cuántos registros hay guardados en SQLite

consulta = """
SELECT COUNT(*) AS total_registros
FROM trafico
"""

pd.read_sql_query(consulta, conexion)

,total_registros
0,4493


In [25]:
# Preparar los datos para insertar en SQLite
columnas_db = [
    "idelem",
    "fecha_hora",
    "descripcion",
    "accesoAsociado",
    "intensidad",
    "ocupacion",
    "carga",
    "nivelServicio",
    "intensidadSat",
    "error",
    "subarea",
    "st_x",
    "st_y",
    "estado_trafico"
]

# Insertar los registros ignorando los que ya existen
sql = """
INSERT OR IGNORE INTO trafico (
    idelem, fecha_hora, descripcion, accesoAsociado,
    intensidad, ocupacion, carga, nivelServicio,
    intensidadSat, error, subarea, st_x, st_y,
    estado_trafico
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

datos = df_trafico_limpio[columnas_db].copy()

# SQLite necesita recibir la fecha como texto
datos["fecha_hora"] = datos["fecha_hora"].astype(str)

conexion.executemany(
    sql,
    datos.itertuples(index=False, name=None)
)

conexion.commit()

print("Inserción completada")

Inserción completada


In [26]:
# Descargar una nueva captura del tráfico

response_nueva = requests.get(url)

print("Status:", response_nueva.status_code)

root_nuevo = ET.fromstring(response_nueva.content)

fecha_hora_nueva = root_nuevo.find("fecha_hora").text

print("Nueva fecha/hora:", fecha_hora_nueva)

Status: 200
Nueva fecha/hora: 26/08/2026 20:00:04


In [27]:
# Convertir la nueva captura XML a DataFrame

registros_nuevos = []

for pm in root_nuevo.findall("pm"):
    fila = {
        "fecha_hora": fecha_hora_nueva
    }

    for campo in pm:
        fila[campo.tag] = campo.text

    registros_nuevos.append(fila)

df_nuevo = pd.DataFrame(registros_nuevos)

print("Dimensiones nueva captura:", df_nuevo.shape)
df_nuevo.head()

Dimensiones nueva captura: (4893, 14)


,fecha_hora,idelem,descripcion,accesoAsociado,intensidad,ocupacion,carga,nivelServicio,intensidadSat,error,subarea,st_x,st_y,velocidad
0,26/08/2026 20:00:04,9841,Valle de Mena S-E - Acc.Ramon Castroviejo-Gta....,0301005,320,1,10,0,3100,N,0328,"438339,375874991","4480454,96970565",NaN
1,26/08/2026 20:00:04,9843,Dr.Ramon Castroviejo O-E - Av.Miraflores-San M...,0301003,1020,25,52,2,2000,N,0328,"438098,880458143","4480455,13738494",NaN
2,26/08/2026 20:00:04,9842,Nueva Zelanda S-N - Isla Cristina-Gta.Isaac Rabín,0301001,80,1,19,0,700,N,0328,"438305,950926946","4480365,97449961",NaN
3,26/08/2026 20:00:04,11396,Dr. Ramon Castroviejo E-O - Islas Bikini-Gta I...,0301004,700,1,17,0,3150,N,0328,"438345,801923105","4480548,97911408",NaN
4,26/08/2026 20:00:04,11243,Islas Aleutianas N-S - Manuel Garrido -Isaac R...,0301002,100,0,7,0,1350,N,0328,"438226,45123268","4480578,31957825",NaN
